# Day 30: Agent Architectures & ReAct

**Week 5 — Agent Development**

---

## 📚 Theory: What Makes an LLM an "Agent"?

An LLM alone is just a text generator. It becomes an **Agent** when it is given a loop, a scratchpad (memory), and the ability to execute external tools to alter its environment. 

### The ReAct Pattern (Reasoning + Acting)
Introduced by Google Brain and Princeton researchers, ReAct forces the LLM to alternate between thinking about what to do, and actually doing it. 

The standard loop looks like this:
1. **Thought**: *"I need to find the current weather in Tokyo to answer the user."*
2. **Action**: `get_weather(location="Tokyo")`
3. **Observation**: *"72 degrees and sunny."*
4. **Thought**: *"I have the weather. I can now answer the user."*
5. **Final Answer**: *"It is currently 72 degrees and sunny in Tokyo!"*

### Multi-Agent Architectures
Complex tasks (like "Build a web app") shouldn't be handled by one massive prompt. Instead, we use multiple specialized agents:
- **Supervisor Agent**: Takes the user request, breaks it into steps, and delegates to workers.
- **Worker Agents**: Specialized personas (e.g., "Coder Agent", "QA Agent", "Researcher Agent").
- **State Machine Routing**: Using libraries like LangGraph or Agno to define deterministic paths between agents (e.g., *Coder output -> QA input. If QA fails -> route back to Coder*).

## 🔨 Exercise 1: Simulating a ReAct Loop

Let's write a mock ReAct loop orchestrator in pure Python to understand how libraries like LangChain or Agno work under the hood.

**Task**: Complete the `run_react_agent` function. It should `while` loop, asking the `mock_llm` for the next step. If it returns an `Action`, execute the tool and append the `Observation`. If it returns a `Final Answer`, break the loop.

In [ ]:
import re

def mock_tool_weather(location: str) -> str:
    if location == "Tokyo": return "72F, Sunny"
    return "Unknown"

# Mock LLM that has a pre-programmed script for this exercise
class MockLLM:
    def __init__(self):
        self.step = 0
        
    def generate(self, prompt: str) -> str:
        if self.step == 0:
            self.step += 1
            return "Thought: I need to check the weather in Tokyo.\nAction: get_weather(Tokyo)"
        else:
            return "Thought: I have the information.\nFinal Answer: It's 72F and Sunny in Tokyo."

def run_react_agent(user_query: str) -> str:
    llm = MockLLM()
    prompt = f"User: {user_query}\n"
    
    # YOUR CODE HERE
    # 1. Loop max 5 times to prevent infinite loops
    # 2. Call llm.generate(prompt)
    # 3. Check if "Final Answer:" is in the response. If so, return it.
    # 4. Check if "Action:" is in the response. If so, parse the tool and location, call the tool, and append "Observation: {result}\n" to the prompt.
    pass

# Test
# print(run_react_agent("What is the weather in Tokyo?"))

In [ ]:
# Solution
def run_react_agent_solution(user_query: str) -> str:
    llm = MockLLM()
    prompt = f"User: {user_query}\n"
    
    for i in range(5):  # Max steps
        print(f"--- Step {i+1} ---")
        response = llm.generate(prompt)
        print(response)
        prompt += response + "\n"
        
        if "Final Answer:" in response:
            return response.split("Final Answer:")[1].strip()
            
        if "Action:" in response:
            # Extract tool call (e.g. "get_weather(Tokyo)")
            # Simplified parsing for the mock
            match = re.search(r'Action: get_weather\((.*?)\)', response)
            if match:
                location = match.group(1)
                obs = mock_tool_weather(location)
                print(f"Observation: {obs}")
                prompt += f"Observation: {obs}\n"
                
    return "Error: Max steps reached"

print("\nSolution Execution:")
final = run_react_agent_solution("What is the weather in Tokyo?")
print(f"\nResult: {final}")


## 🔨 Exercise 2: Defining a State Machine

In modern agent frameworks (like LangGraph), agent workflows are defined as graphs. Nodes are functions/agents, and Edges are conditional routing logic based on the state.

**Task**: Create a simple routing function `route_task(state)` that takes a dictionary `state = {'code_written': True, 'tests_passed': False}`. 
- If code is not written, route to `"CODER"`.
- If code is written but tests failed, route to `"DEBUGGER"`.
- If tests passed, route to `"END"`.

In [ ]:
def route_task(state: dict) -> str:
    """Return the name of the next node/agent."""
    # YOUR CODE HERE
    pass

assert route_task({'code_written': False, 'tests_passed': False}) == "CODER"
assert route_task({'code_written': True, 'tests_passed': False}) == "DEBUGGER"
assert route_task({'code_written': True, 'tests_passed': True}) == "END"
print("All tests passed! ✅")

In [ ]:
# Solution
def route_task_solution(state: dict) -> str:
    if not state.get('code_written', False):
        return "CODER"
    elif not state.get('tests_passed', False):
        return "DEBUGGER"
    else:
        return "END"


## 📝 Day 30 Review

### Concepts Validated Today
- [ ] The **ReAct (Reasoning + Acting)** loop that forms the core of autonomous agents.
- [ ] Parsing "Thoughts" and "Actions" from LLM text streams to trigger external python functions.
- [ ] How Multi-Agent systems are simply **State Machines** where the output of one agent dictates the routing to the next.

### Tomorrow's Preview
**Day 31: Google ADK Deep Dive** — We will explore the specific components likely found in Google's proprietary Agent Development Kit and how to design APIs that interface directly with Gemini.